# NetworkEvolveLab: Quickstart Tutorial

Welcome to **NetworkEvolveLab**! This notebook demonstrates how to load real-world empirical social networks, simulate the dynamic co-evolution of agent cognitive states and social topology (selective unfollowing and blocking), and observe the emergence of echo chambers and topological phase transitions.

In [ ]:
# 1. Import core library
import network_evolve as ne
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

print(f"NetworkEvolveLab Version: {ne.__version__}")

## 2. Load Empirical Benchmark Network
We load the **Twitter 2020 US Presidential Election** partisan retweet network along with empirical DW-NOMINATE ideological coordinates.

In [ ]:
g, ideology = ne.load_twitter_election(num_nodes=60, seed=42)
print(f"Nodes: {g.number_of_nodes()}, Edges: {g.number_of_edges()}")
print(f"Sample belief positions: {[round(ideology[i], 3) for i in range(5)]}")

## 3. Initialize Heterogeneous Agents & CoEvolutionEngine
We configure three behavioral archetypes:
- **Hub Broadcasters** (Top 2%)
- **Active Bridgers** (8.5% with wider tolerance)
- **Long-tail Responders** (89.5%)
and enable the bilateral tie-severing block mechanism (`block_diff = 1.20`).

In [ ]:
agents = {}
num_nodes = g.number_of_nodes()
for i in range(num_nodes):
    if i < 2:
        arch, tol = "hub_broadcaster", 0.40
    elif 2 <= i < 7:
        arch, tol = "active_bridger", 0.65
    else:
        arch, tol = "long_tail_responder", 0.40
        
    agents[i] = ne.EvolvingAgent(
        agent_id=i,
        archetype=arch,
        belief=ideology[i],
        tolerance=tol,
        damping=0.10
    )

engine = ne.CoEvolutionEngine(
    graph=ne.DynamicSocialGraph(initial_graph=g),
    agents=agents,
    block_diff=1.20,
    recommendation_friction=1.0
)
print("Engine initialized successfully.")

## 4. Run Co-evolution Simulation Loop
We track Newman modularity $Q$, opinion polarization variance $\Pi$, and remaining edge counts over 20 rounds of interaction.

In [ ]:
history_q = []
history_pi = []
history_edges = []

for r in range(20):
    engine.step()
    metrics = ne.snapshot(engine)
    history_q.append(metrics["modularity_q"])
    history_pi.append(metrics["polarization"])
    history_edges.append(metrics["num_edges"])
    if (r + 1) % 5 == 0 or r == 0:
        print(f"Round {r+1:02d}: Modularity Q = {metrics['modularity_q']:.3f}, Polarization Pi = {metrics['polarization']:.3f}, Edges = {metrics['num_edges']}")

## 5. Visualize Dynamic Topology Evolution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

rounds = np.arange(1, len(history_q) + 1)
ax1.plot(rounds, history_q, 'o-', color='#1f77b4', linewidth=2, label='Newman Modularity Q')
ax1.plot(rounds, history_pi, 's--', color='#d62728', linewidth=2, label='Polarization Index Pi')
ax1.axhline(0.500, color='gray', linestyle=':', label='Theoretical Bipolar Bound (Q <= 0.50)')
ax1.set_xlabel('Simulation Round')
ax1.set_ylabel('Metric Value')
ax1.set_title('Co-evolution of Cognition & Modularity')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

ax2.plot(rounds, history_edges, '^-', color='#2ca02c', linewidth=2, label='Active Directed Edges')
ax2.set_xlabel('Simulation Round')
ax2.set_ylabel('Edge Count')
ax2.set_title('Tie Severance & Selective Unfollowing')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()